<a href="https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-coder874/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [26]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last90,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev90,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last90,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last90

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 180 DAY

        GROUP BY 1,2

        HAVING imp_prev90 >= 300
    )

    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

94,366 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90
0,client_e547b89c05043229,content_b962dd8115b75719,1657.0,1684.0,2.0,12.171323
1,client_e547b89c05043229,content_d606939ec54831d3,1486.0,1633.0,2.0,38.192883
2,client_e547b89c05043229,content_c06834d9f426a3d6,412.0,376.0,1.0,32.412067
3,client_e547b89c05043229,content_d812cb8e421e99d1,3115.0,4537.0,10.0,9.428185
4,client_e547b89c05043229,content_8e30813d22a7b445,936.0,1039.0,1.0,31.915502


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [27]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


joined: 94,366 rows


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,pos_last90,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_b962dd8115b75719,1657.0,1684.0,2.0,12.171323,11.0,0.142426,0.727821,56.0,215.0,0.260465
1,client_e547b89c05043229,content_d606939ec54831d3,1486.0,1633.0,2.0,38.192883,8.0,0.253701,0.583445,66.0,242.0,0.272727
2,client_e547b89c05043229,content_c06834d9f426a3d6,412.0,376.0,1.0,32.412067,2.0,0.271845,0.677184,11.0,21.0,0.523810
3,client_e547b89c05043229,content_d812cb8e421e99d1,3115.0,4537.0,10.0,9.428185,12.0,0.067416,0.843660,53.0,277.0,0.191336
4,client_e547b89c05043229,content_8e30813d22a7b445,936.0,1039.0,1.0,31.915502,2.0,0.033120,0.925214,27.0,39.0,0.692308


In [9]:
data[[
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share'
]] = data[[
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share'
]].fillna(0)

print(data[['visible_queries','rare_share','anon_share','top_query_share']].isna().sum())

visible_queries    0
rare_share         0
anon_share         0
top_query_share    0
dtype: int64


In [13]:
print("data shape:", data.shape)
print("features shape:", features.shape)
print(features.head())

data shape: (0, 13)
features shape: (0, 6)
Empty DataFrame
Columns: [client_hash_id, content_hash_id, imp_last30, imp_prev30, clk_last30, pos_last30]
Index: []


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [29]:
data['is_declining'] = (data['imp_last90'] < 0.8 * data['imp_prev90']).astype(int)

feature_cols = [
    'imp_prev90',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'pos_last90'
]

model_data = data.copy()

model_data[feature_cols] = model_data[feature_cols].fillna(0)

print("Rows for model:", len(model_data))

X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
).fit(X_tr, y_tr)

print(f'base rate: {max(y_te.mean(), 1-y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))

Rows for model: 94366
base rate: 0.527
              precision    recall  f1-score   support

           0      0.867     0.839     0.853     11157
           1      0.860     0.885     0.872     12435

    accuracy                          0.863     23592
   macro avg      0.864     0.862     0.863     23592
weighted avg      0.863     0.863     0.863     23592



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [18]:
print(data.shape)
print(data.head())

(111247, 13)
            client_hash_id           content_hash_id  imp_last30  imp_prev30  \
0  client_e547b89c05043229  content_6b80dfab2e0ffa2e      1110.0       955.0   
1  client_e547b89c05043229  content_d7bb60ec9a42c11a      3735.0      3338.0   
2  client_e547b89c05043229  content_401dcc5cd616e3dd       181.0       130.0   
3  client_e547b89c05043229  content_18d95bd7890430ed       151.0       340.0   
4  client_e547b89c05043229  content_56f46c55f0348ab4       392.0       531.0   

   clk_last30  pos_last30  visible_queries  rare_share  anon_share  \
0        12.0    7.543789              1.0    0.022750    0.957216   
1        33.0    5.446636             14.0    0.017946    0.932994   
2         0.0    6.874167              3.0    0.162037    0.552469   
3         0.0   33.665367              2.0    0.108932    0.820261   
4         3.0   12.995100              5.0    0.163052    0.788332   

   top_query_impressions  kept_impressions  top_query_share  is_declining  
0        

In [36]:
print("feature_cols:", len(feature_cols))
print("model features:", len(model.feature_importances_))

feature_cols: 7
model features: 6


In [41]:
import pandas as pd

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
0,imp_prev90,0.302977
1,visible_queries,0.179407
2,rare_share,0.155451
3,anon_share,0.143342
5,pos_last90,0.122615
4,top_query_share,0.096207


In [20]:
data["position_signal"] = data["pos_last30"]

In [21]:
feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'position_signal'
]

In [24]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'position_signal'
]

model_data = data.copy()
model_data[feature_cols] = model_data[feature_cols].fillna(0)

X = model_data[feature_cols]
y = model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_tr, y_tr)

print(classification_report(
    y_te,
    model.predict(X_te),
    digits=3
))

              precision    recall  f1-score   support

           0      0.664     0.490     0.564      8045
           1      0.812     0.898     0.853     19675

    accuracy                          0.780     27720
   macro avg      0.738     0.694     0.708     27720
weighted avg      0.769     0.780     0.769     27720



In [25]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
5,position_signal,0.282706
0,imp_prev30,0.181370
2,rare_share,0.159381
3,anon_share,0.156511
4,top_query_share,0.128998
1,visible_queries,0.091033


In [30]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
0,imp_prev90,0.302977
1,visible_queries,0.179407
2,rare_share,0.155451
3,anon_share,0.143342
5,pos_last90,0.122615
4,top_query_share,0.096207


In [39]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# prepare data
model_data = data.copy()

feature_cols = [
    'imp_prev90',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'pos_last90'
]

model_data[feature_cols] = model_data[feature_cols].fillna(0)


# split by CLIENT, not random rows
splitter = GroupShuffleSplit(
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)


X_train = model_data.iloc[train_idx][feature_cols]
X_test = model_data.iloc[test_idx][feature_cols]

y_train = model_data.iloc[train_idx]["is_declining"]
y_test = model_data.iloc[test_idx]["is_declining"]


print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 74571
Test rows: 19795


In [40]:
model_client = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_client.fit(
    X_train,
    y_train
)


pred = model_client.predict(X_test)

print(classification_report(
    y_test,
    pred,
    digits=3
))

              precision    recall  f1-score   support

           0      0.853     0.844     0.849      9326
           1      0.862     0.871     0.867     10469

    accuracy                          0.858     19795
   macro avg      0.858     0.857     0.858     19795
weighted avg      0.858     0.858     0.858     19795



In [42]:
print("""
MODEL SUMMARY

Goal:
Predict whether a content item will have an impressions decline.

Dataset:
FlyRank full warehouse release (~79M daily rows)

Features used:
- Previous 90 day impressions
- Visible queries
- Rare query share
- Anonymized impression share
- Top query concentration
- Average position

Model:
Random Forest Classifier

Validation:
GroupShuffleSplit by client_hash_id

Final Result:
Accuracy: 85.8%
Macro F1: 0.858

Conclusion:
The model performs similarly on unseen clients,
showing that the learned patterns generalize.
""")


MODEL SUMMARY

Goal:
Predict whether a content item will have an impressions decline.

Dataset:
FlyRank full warehouse release (~79M daily rows)

Features used:
- Previous 90 day impressions
- Visible queries
- Rare query share
- Anonymized impression share
- Top query concentration
- Average position

Model:
Random Forest Classifier

Validation:
GroupShuffleSplit by client_hash_id

Final Result:
Accuracy: 85.8%
Macro F1: 0.858

Conclusion:
The model performs similarly on unseen clients,
showing that the learned patterns generalize.

